# Part 4: Deep Learning Methods for Time Series Analysis

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Deep Learning Methods for Time Series Analysis</h3>
</div>

In this notebook, we use **deep learning** models to forecast a real-world time series.

We focus on four approaches:

1. Dense Neural Network (MLP baseline)
2. LSTM
3. GRU
4. 1D CNN


<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">Learning Goals</h3>
</div>

By the end of this notebook, you will be able to:

- prepare a real time series for supervised deep learning
- create sliding-window sequences
- train and compare multiple deep learning architectures
- evaluate models using MAE and RMSE
- visualize forecast performance on the holdout test period

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">0. Install Required Libraries (if needed)</h3>
</div>

Run this cell only if your environment is missing required packages.

We use:

- `tensorflow` for deep learning
- `statsmodels` to load a real dataset
- `scikit-learn` for scaling and metrics

In [ ]:
# Uncomment if needed:
# !pip install tensorflow statsmodels scikit-learn matplotlib pandas numpy

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">1. Import Libraries</h3>
</div>

We import data tools, plotting tools, preprocessing utilities, and Keras model components.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import statsmodels.api as sm

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

np.random.seed(42)
tf.random.set_seed(42)

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">2. Load a Real Time Series Dataset</h3>
</div>

We use the **Mauna Loa atmospheric CO2** dataset from `statsmodels`.

Why this is a good real-world dataset:

- It measures atmospheric CO2 concentration over time.
- It contains clear trend and seasonality.
- It is frequently used for forecasting demonstrations.

The raw dataset is weekly with some missing values, so we resample to monthly frequency and interpolate missing points.

In [ ]:
co2_raw = sm.datasets.co2.load_pandas().data

# Convert to a clean monthly series
co2_monthly = co2_raw['co2'].resample('MS').mean().interpolate()

df = pd.DataFrame({
    'ds': co2_monthly.index,
    'y': co2_monthly.values
}).dropna().reset_index(drop=True)

print('Rows:', len(df))
df.head()

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">3. Visualize the Time Series</h3>
</div>

Before modeling, we inspect the series to understand behavior.

Look for:

- long-term trend
- repeating seasonal cycles
- abrupt anomalies

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['ds'], df['y'])
plt.title('Monthly Atmospheric CO2 (Mauna Loa)')
plt.xlabel('Date')
plt.ylabel('CO2 (ppm)')
plt.grid(alpha=0.3)
plt.show()

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">4. Train/Test Split (Time-Aware)</h3>
</div>

For forecasting, we must preserve time order.

We keep the final 24 months as the test set and train on earlier observations only.

In [ ]:
test_horizon = 24
train_df = df.iloc[:-test_horizon].copy()
test_df = df.iloc[-test_horizon:].copy()

print('Train size:', len(train_df))
print('Test size:', len(test_df))

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">5. Scale Data and Build Sliding Windows</h3>
</div>

Neural networks train better when numeric values are scaled.

We use `MinMaxScaler` fit on training data only to avoid leakage.

Then we convert the series into supervised samples using a lookback window: each input uses the previous 12 months to predict the next month.

In [ ]:
lookback = 12

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[['y']])
full_scaled = scaler.transform(df[['y']])

def make_sequences(values, window):
    X, y = [], []
    for i in range(window, len(values)):
        X.append(values[i-window:i, 0])
        y.append(values[i, 0])
    return np.array(X), np.array(y)

X_all, y_all = make_sequences(full_scaled, lookback)
dates_all = df['ds'].iloc[lookback:].reset_index(drop=True)

X_all.shape, y_all.shape

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">6. Build Final Train/Test Tensors</h3>
</div>

Because sequence construction consumes the first 12 observations, we align the train/test split with sequence indices.

We prepare:

- 2D inputs for Dense model: `(samples, lookback)`
- 3D inputs for sequence models: `(samples, lookback, 1)`

In [ ]:
split_date = test_df['ds'].iloc[0]
train_mask = dates_all < split_date
test_mask = dates_all >= split_date

X_train_2d, y_train = X_all[train_mask], y_all[train_mask]
X_test_2d, y_test = X_all[test_mask], y_all[test_mask]
test_dates = dates_all[test_mask].reset_index(drop=True)

X_train_3d = X_train_2d[..., np.newaxis]
X_test_3d = X_test_2d[..., np.newaxis]

print('Dense train:', X_train_2d.shape, y_train.shape)
print('Dense test :', X_test_2d.shape, y_test.shape)
print('Seq train  :', X_train_3d.shape, y_train.shape)
print('Seq test   :', X_test_3d.shape, y_test.shape)

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">7. Define a Reusable Training Helper</h3>
</div>

To keep experiments consistent, we use one helper function for training each model.

This applies:

- Adam optimizer
- MSE loss
- EarlyStopping to reduce overfitting

In [ ]:
def train_model(model, X_train, y_train, model_name, epochs=80, batch_size=16):
    model.compile(optimizer='adam', loss='mse')
    es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[es]
    )
    print(f"{model_name} best val_loss: {np.min(history.history['val_loss']):.6f}")
    return history

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">8. Model A: Dense Neural Network (MLP Baseline)</h3>
</div>

This model treats the previous 12 values as regular numeric features.

It does not explicitly model sequence dynamics, but serves as a useful baseline.

In [ ]:
dense_model = models.Sequential([
    layers.Input(shape=(lookback,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

_ = train_model(dense_model, X_train_2d, y_train, 'Dense')

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">9. Model B: LSTM</h3>
</div>

LSTM is designed for sequential dependencies and long-term memory.

This is often a strong baseline for univariate time series forecasting.

In [ ]:
lstm_model = models.Sequential([
    layers.Input(shape=(lookback, 1)),
    layers.LSTM(64),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

_ = train_model(lstm_model, X_train_3d, y_train, 'LSTM')

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">10. Model C: GRU</h3>
</div>

GRU is a gated recurrent model similar to LSTM but with fewer parameters.

It can train faster while still capturing temporal dependencies.

In [ ]:
gru_model = models.Sequential([
    layers.Input(shape=(lookback, 1)),
    layers.GRU(64),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

_ = train_model(gru_model, X_train_3d, y_train, 'GRU')

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">11. Model D: 1D CNN</h3>
</div>

A 1D CNN learns local temporal patterns using convolution filters.

For many forecasting tasks, CNNs can be competitive and efficient.

In [ ]:
cnn_model = models.Sequential([
    layers.Input(shape=(lookback, 1)),
    layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

_ = train_model(cnn_model, X_train_3d, y_train, 'CNN1D')

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">12. Evaluate All Models</h3>
</div>

Predictions are generated on the test set, then transformed back to original CO2 units.

We report:

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)

In [ ]:
def invert_scale(arr):
    return scaler.inverse_transform(arr.reshape(-1, 1)).ravel()

y_test_real = invert_scale(y_test)

pred_dense = invert_scale(dense_model.predict(X_test_2d, verbose=0).ravel())
pred_lstm = invert_scale(lstm_model.predict(X_test_3d, verbose=0).ravel())
pred_gru = invert_scale(gru_model.predict(X_test_3d, verbose=0).ravel())
pred_cnn = invert_scale(cnn_model.predict(X_test_3d, verbose=0).ravel())

def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

results = []
for name, pred in [
    ('Dense', pred_dense),
    ('LSTM', pred_lstm),
    ('GRU', pred_gru),
    ('CNN1D', pred_cnn),
]:
    mae, rmse = metrics(y_test_real, pred)
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse})

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
results_df

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">13. Visual Comparison of Forecasts</h3>
</div>

A line plot helps us inspect whether each model captures trend and seasonal behavior over the 24-month test horizon.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test_real, label='Actual', linewidth=2)
plt.plot(test_dates, pred_dense, label='Dense', alpha=0.9)
plt.plot(test_dates, pred_lstm, label='LSTM', alpha=0.9)
plt.plot(test_dates, pred_gru, label='GRU', alpha=0.9)
plt.plot(test_dates, pred_cnn, label='CNN1D', alpha=0.9)

plt.title('Deep Learning Forecast Comparison (CO2)')
plt.xlabel('Date')
plt.ylabel('CO2 (ppm)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

<div style="
    background-color:#8F0177;
    padding:15px;
    border-radius:8px;
    color:white;
    display:flex;
    align-items:center;
">
    <h3 style="margin:0;">14. Summary</h3>
</div>

In this notebook, we demonstrated multiple deep learning methods for time series forecasting on a real dataset.

Key takeaways:

- Time order must be preserved in train/test splitting.
- Sliding-window framing is essential for supervised deep learning.
- Recurrent models (LSTM/GRU) and CNNs can model temporal patterns effectively.
- Model comparison should include both numeric metrics and visual inspection.